In [194]:
%%time
import warnings
import optuna
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline,make_union
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures, PowerTransformer
from sklearn.feature_selection import SelectKBest, RFECV, SelectFromModel, mutual_info_regression
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import mean_absolute_error, mean_squared_error, make_scorer
from sklearn.model_selection import cross_val_score, KFold, ShuffleSplit, learning_curve

warnings.filterwarnings('ignore')

Wall time: 16.4 s


In [195]:
df=pd.read_csv("training_dataset.csv")
df.drop(["region__peak_sales_dollar_amt_per_hour_v2","region__peak_returns_dollar_amt_per_hour_v2","Unnamed: 0","observation_id","observation_timestamp"], axis=1, inplace=True)
df.head()

,hour_of_day,register__sales_dollar_amt_this_hour,register__payment_types_accepted,register__peak_sales_dollar_amt_per_hour,register__sales_dollar_amt_last_hour,register__sales_quantity_last_hour,register__sales_quantity_rescanned_frac,register__sales_payments_declined_frac,register__peak_returns_dollar_amt_per_hour,register__returns_dollar_amt_last_hour,...,region__mean_service_time_per_customer,region__stdev_service_time_per_customer,region__sales_dollar_amt_last_hour,region__returns_dollar_amt_last_hour,region__nighttime_open_registers,region__nighttime_service_time_per_customer,region__nighttime_sales_amt_per_hour,region__nighttime_returns_amt_per_hour,region__peak_sales_dollar_amt_per_hour,region__peak_returns_dollar_amt_per_hour
0,15,347.29,Cash+Credit,-0.7383,-0.1270,-0.1993,-0.8299,-0.1247,-0.5721,-0.2582,...,-0.9255,-0.3891,-0.6920,-0.4605,-0.5180,-1.0062,-0.6462,-0.6030,-0.4773,-1.7951
1,21,361.59,Cash+Credit,0.6483,-0.0362,-0.0777,-0.7395,-0.1135,-0.5251,-0.2157,...,0.7815,0.0663,-0.6531,-0.4434,-0.6498,0.9031,-0.6493,-0.6106,0.4998,0.8939
2,11,850.73,Cash+Credit,-0.4950,-0.1268,-0.1974,1.3139,0.1075,-0.5251,-0.2579,...,-1.2559,-0.4826,-0.6120,-0.1786,-0.6040,-1.4229,-0.6456,-0.6037,1.9337,2.4046
3,10,1175.69,Cash+Credit,-0.5594,-0.1270,-0.1991,-0.8299,-0.1247,-0.5251,-0.2582,...,-1.2280,-0.4809,-0.5472,0.1246,-0.5925,-1.4097,-0.6478,-0.6079,0.1288,-0.4983
4,14,3204.53,Cash+Credit,0.5693,-0.1221,-0.1632,-0.7071,-0.1247,0.7604,-0.2565,...,0.2442,-0.3021,-0.3838,0.1996,-0.5696,1.1231,-0.6475,-0.6032,1.0050,1.5660


In [196]:
def regression_accuracy(y_meas, y_pred, max_error=20, error_type='relative'):
    '''Compares predicted & measured values, returning the percentage of predictions
       that are within a set error limit. This error limit can be an absolute value
       or a relative percentage'''

    # OPTION 1: Relative percentage
    if ( error_type == 'relative' ):
        mask = 100.0 * abs((y_pred - y_meas) / y_meas) < max_error

    # OPTION 2: Absolute value
    if ( error_type == 'absolute' ):
        mask = abs(y_pred - y_meas) < max_error

    accuracy = sum(mask) / len(mask)

    return accuracy


scorer=make_scorer(regression_accuracy)

In [197]:
def imputation(df):
    return df

In [198]:
def encodage(df):
    return df

In [199]:
def pre_processing(df):
    #df=imputation(df)
    df=encodage(df)

    if "register__sales_dollar_amt_this_hour" in df.columns.values:
        X = df.drop("register__sales_dollar_amt_this_hour", axis=1)
        y = df.register__sales_dollar_amt_this_hour
        return X,y
    else:
        X = df
        return X

In [200]:
def make_model(df, estimator=RandomForestRegressor(random_state=0)):
    numerical_features=df.select_dtypes(["float","int"]).columns

    numerical_pipeline=make_pipeline(PowerTransformer())
    
    process=make_column_transformer((numerical_pipeline, numerical_features))

    model=make_pipeline(process, estimator)

    return model

In [201]:
def regression_accuracy_lgb(y_pred, y_meas):
    '''Compares predicted & measured values, returning the percentage of predictions
       that are within a set error limit. This error limit can be an absolute value
       or a relative percentage'''
    # y_meas = data.get_label()
    # weight = data.get_weight()
    max_error=20
    error_type='relative'
    # OPTION 1: Relative percentage
    if ( error_type == 'relative' ):
        mask = 100.0 * abs((y_pred - y_meas) / y_meas) < max_error

    # OPTION 2: Absolute value
    if ( error_type == 'absolute' ):
        mask = abs(y_pred - y_meas) < max_error

    reg_accuracy = sum(mask) / len(mask)
    #reg_accuracy=np.average(reg_accuracy, weights=weight)

    return "regression_accuracy_lgb", reg_accuracy, True

In [202]:
with open("used_columns.txt","r") as file:
    used_columns=[i.replace("\n","") for i in file.readlines()]

In [203]:
# used_columns.extend(df.select_dtypes("object").columns)
# used_columns.append("store__type_code")
train_df=df[used_columns]
train,test=train_test_split(train_df, test_size=0.2)
#X_train,y_train=pre_processing(train)
#X_test,y_test=pre_processing(test)

X=df[used_columns]
y=df.register__sales_dollar_amt_this_hour

In [204]:
RANDOM_SEED=0
NUM_THREADS=0
MODEL_PARAMS_LGB = {
    "objective": "regression",  # <= set objective
    "metric":None,
    "seed": RANDOM_SEED,
    "verbose": -1
}   
FIT_PARAMS_LGB = {"early_stopping_rounds": 100, "verbose_eval":100}

cv=KFold(n_splits=5,shuffle=True,random_state=0)
scores=[]
for i,j in enumerate(cv.split(X,y)):
    model=lgb.LGBMRegressor(boosting_type='gbdt', objective='regression',metric=None)
    train_X=PowerTransformer().fit_transform(X.iloc[j[0]])
    train_y=y.iloc[j[0]]
    test_X=PowerTransformer().fit_transform(X.iloc[j[1]])
    test_y=y.iloc[j[1]]
    model.fit(train_X,train_y,
        eval_names=['train', 'valid'],
        eval_set=[(train_X,train_y),(test_X, test_y)], 
        eval_metric=regression_accuracy_lgb,
        verbose = False)
    
    y_pred=model.predict(X.iloc[j[1]])
    score=regression_accuracy(y_pred,y.iloc[j[1]])
    scores.append(score)
    print(f"Fold {i}\t:{score}\n")

print(f"CV_SCORES_mean: {np.mean(scores)}")
print(f"CV_SCORES_std: {np.std(scores)}")

# lgb_tr = lgb.Dataset(X_train, y_train)
# lgb_val = lgb.Dataset(X_test, y_test)

# model=lgb.train(
#     MODEL_PARAMS_LGB,
#     lgb_tr,
#     **FIT_PARAMS_LGB,
#     valid_names=['train', 'valid'], 
#     valid_sets=[lgb_tr, lgb_val],
#     feval= regression_accuracy_lgb
# )


SyntaxError: invalid syntax (Temp/ipykernel_10376/952696066.py, line 14)

In [207]:
def objective(trial):
    
    params = {
        "boosting_type": trial.suggest_categorical("boosting_type", choices=["gbdt", "rf", "dart"]),
        "objective": "regression",
        "metric":None,
        "seed": RANDOM_SEED,
        "verbose": -1,
        'lambda_l1': trial.suggest_loguniform('lambda_l1', 1e-8, 10.0),
        'lambda_l2': trial.suggest_loguniform('lambda_l2', 1e-8, 10.0),
        'num_leaves': trial.suggest_int('num_leaves', 2, 256),
        'feature_fraction': trial.suggest_uniform('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_uniform('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        "early_stopping_rounds": 100,
    }

    scores=[]
    for i,j in enumerate(cv.split(X,y)):
        model=lgb.LGBMRegressor(**params)
        train_X=X.iloc[j[0]]
        train_y=y.iloc[j[0]]
        test_X=X.iloc[j[1]]
        test_y=y.iloc[j[1]]
        model.fit(train_X,train_y,
            eval_names=['train', 'valid'],
            eval_set=[(train_X,train_y),(test_X, test_y)], 
            eval_metric=regression_accuracy_lgb,
            verbose = False)
        
        y_pred=model.predict(X.iloc[j[1]])
        score=regression_accuracy(y_pred,y.iloc[j[1]])
        scores.append(score)
    
    return np.mean(scores)

In [208]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100, n_jobs=-1, timeout=3200)

[I 2022-09-16 07:36:56,953] A new study created in memory with name: no-name-c1fa93ab-998f-4f7f-acf4-a4b1674d18a8
[I 2022-09-16 07:39:37,031] Trial 3 finished with value: 0.5466182383404872 and parameters: {'boosting_type': 'dart', 'lambda_l1': 0.07784940374961027, 'lambda_l2': 0.0001350608864609646, 'num_leaves': 37, 'feature_fraction': 0.47788004421646113, 'bagging_fraction': 0.4770476698560734, 'bagging_freq': 7, 'min_child_samples': 34}. Best is trial 3 with value: 0.5466182383404872.
[I 2022-09-16 07:39:45,163] Trial 1 finished with value: 0.8599288646042099 and parameters: {'boosting_type': 'rf', 'lambda_l1': 7.162535554865133e-07, 'lambda_l2': 0.18946272323158922, 'num_leaves': 208, 'feature_fraction': 0.4929438609144293, 'bagging_fraction': 0.7334733217189013, 'bagging_freq': 7, 'min_child_samples': 19}. Best is trial 1 with value: 0.8599288646042099.
[I 2022-09-16 07:39:45,829] Trial 2 finished with value: 0.8568494188936416 and parameters: {'boosting_type': 'gbdt', 'lambda_l1